# Chapter 13: Retrieval-Augmented Generation (RAG)
## Notebook 03 — Advanced RAG

This notebook covers the techniques that separate a *demo* RAG from a *production* RAG:

| Topic | Section |
|-------|--------|
| Hybrid search: dense + BM25 with reciprocal rank fusion | §1 |
| Query rewriting / HyDE / multi-query | §2 |
| Evaluation: faithfulness, answer relevance, context precision/recall | §3 |
| Agentic and multi-hop retrieval | §4 |
| Production: latency, caching, freshness, sharding, cost | §5 |
| Capstone design exercise | §6 |

**Estimated time:** 2 hours

---
*Generated by Berta AI*

In [ ]:
import sys, os, re, json, time
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

# Load the corpus
CORPUS_PATH = os.path.join('..', 'datasets', 'sample_corpus.txt')
with open(CORPUS_PATH) as f:
    raw = f.read()

pattern = re.compile(r'^\[(doc-\d+)\]\s*(.+)$', re.MULTILINE | re.DOTALL)
documents = {}
for para in re.split(r'\n\s*\n', raw):
    m = pattern.match(para.strip())
    if m:
        documents[m.group(1)] = m.group(2).strip()
print("Loaded", len(documents), "documents")

---
## 1. Hybrid Search

Dense retrievers excel at *semantic* matches ("autocar" ↔ "vehicle"). Sparse retrievers like BM25 excel at *exact* matches (rare entity names, code tokens, identifiers). **Hybrid search** combines them.

We use **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

with `k = 60`. RRF needs no score calibration between rankers.

In [ ]:
from rag_pipeline import TfidfEmbedder
from vectorstore import InMemoryVectorStore, BM25Index, HybridIndex

texts = list(documents.values())
ids = list(documents.keys())

# Dense
embedder = TfidfEmbedder(dim=128).fit(texts)
embs = embedder.encode(texts)
dense = InMemoryVectorStore(dim=embs.shape[1])
dense.add(embeddings=embs, chunk_ids=ids, texts=texts)

# Sparse
sparse = BM25Index()
sparse.add(chunk_ids=ids, texts=texts)

# Hybrid
hybrid = HybridIndex(dense=dense, sparse=sparse, rrf_k=60)
print(f"dense docs={len(dense)}  sparse docs={len(sparse)}")

In [ ]:
def compare_retrievers(query, top_k=3):
    q_emb = embedder.encode_query(query)
    d = dense.search(q_emb, top_k=top_k)
    s = sparse.search(query, top_k=top_k)
    h = hybrid.search(query, q_emb, top_k=top_k)
    print(f"\nQuery: {query}")
    print("  dense :", [(r.chunk_id, round(r.score, 3)) for r in d])
    print("  sparse:", [(r.chunk_id, round(r.score, 3)) for r in s])
    print("  hybrid:", [(r.chunk_id, round(r.score, 3)) for r in h])

for q in ["What is HyDE in retrieval?",
          "What does FAISS provide?",
          "How is faithfulness measured?"]:
    compare_retrievers(q)

---
## 2. Query Rewriting

The user's literal question is rarely the best search query. Three popular rewriters:

- **HyDE** — ask the LLM to draft a *hypothetical* answer, embed that, and search with it.
- **Multi-query** — ask the LLM for *N paraphrases* of the question; retrieve with each; union and rerank.
- **Decomposition** — break a multi-part question into atomic sub-queries.

Below we implement deterministic stubs that mimic these patterns offline.

In [ ]:
def fake_hyde(question, embedder, store, top_k=3):
    """Mock HyDE: 'hallucinate' a one-line answer by repeating keywords, then retrieve."""
    keywords = [w for w in re.findall(r'[A-Za-z]+', question) if len(w) > 3]
    hypo = " ".join(keywords) + " is a technique used in retrieval-augmented generation."
    q_emb = embedder.encode_query(hypo)
    return store.search(q_emb, top_k=top_k), hypo

def multi_query(question, embedder, store, n=3, top_k=3):
    """Mock multi-query: produce paraphrase variants by reordering keywords."""
    words = question.split()
    variants = [question]
    for i in range(1, n):
        if len(words) > 3:
            variants.append(" ".join(words[i:] + words[:i]))
    all_hits = {}
    for v in variants:
        for h in store.search(embedder.encode_query(v), top_k=top_k):
            all_hits.setdefault(h.chunk_id, h)
    return list(all_hits.values())[:top_k], variants

q = "What is HyDE in retrieval?"
hits, hypo = fake_hyde(q, embedder, dense)
print("HyDE hypothesis:", hypo)
print("HyDE retrieved :", [h.chunk_id for h in hits])

mq_hits, variants = multi_query(q, embedder, dense)
print("\nVariants:", variants)
print("Multi-query retrieved:", [h.chunk_id for h in mq_hits])

---
## 3. Evaluation

A complete RAG evaluation answers three questions:

1. **Did we retrieve the right context?** — context precision / recall, hit@k
2. **Is the answer supported by the context?** — *faithfulness*
3. **Does the answer address the question?** — *answer relevance*

Below we compute simple lexical proxies for all four. In production you'd swap these for an LLM-as-judge or a fine-tuned classifier.

In [ ]:
def context_precision(retrieved_ids, gold_ids, k):
    if k == 0:
        return 0.0
    gold = set(gold_ids)
    return sum(1 for r in retrieved_ids[:k] if r in gold) / k

def context_recall(retrieved_ids, gold_ids, k):
    gold = set(gold_ids)
    if not gold:
        return 0.0
    return sum(1 for g in gold if g in retrieved_ids[:k]) / len(gold)

def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / max(1, len(sa | sb))

def faithfulness(answer, contexts):
    """Fraction of answer sentences that share >=2 content words with any context."""
    sents = [s.strip() for s in re.split(r'[.!?]', answer) if s.strip()]
    if not sents:
        return 0.0
    ctx_words = set()
    for c in contexts:
        ctx_words |= {w.lower() for w in c.text.split() if len(w) > 3}
    supported = 0
    for s in sents:
        s_words = {w.lower() for w in s.split() if len(w) > 3}
        if len(s_words & ctx_words) >= 2:
            supported += 1
    return supported / len(sents)

def answer_relevance(answer, question):
    return jaccard(answer, question)

print("Helper metrics defined.")

In [ ]:
# Run evaluation against the chapter's queries.csv
queries_df = pd.read_csv(os.path.join('..', 'datasets', 'queries.csv'))
queries_df['relevant'] = queries_df['relevant_doc_ids'].str.split('|')

results = []
for _, row in queries_df.iterrows():
    q = row['query']
    q_emb = embedder.encode_query(q)
    top = hybrid.search(q, q_emb, top_k=5)
    rids = [h.chunk_id for h in top]
    results.append({
        "query": q,
        "P@3": context_precision(rids, row['relevant'], 3),
        "R@5": context_recall(rids, row['relevant'], 5),
        "MRR": next((1.0 / r for r, rid in enumerate(rids, 1) if rid in row['relevant']), 0.0),
    })

df = pd.DataFrame(results)
print(df.head(10))
print("\nMacro-averages:")
print(df[['P@3', 'R@5', 'MRR']].mean().round(3))

In [ ]:
# Run end-to-end answers and compute faithfulness + answer relevance
from rag_pipeline import RAGPipeline, MockLLM
from chunking import Chunker

pipe = RAGPipeline(
    chunker=Chunker(strategy="sentence", max_tokens=80),
    embedder=embedder,
    llm_client=MockLLM(),
    top_k=3,
)
pipe.index_documents(documents)

with open(os.path.join('..', 'datasets', 'qa_pairs.json')) as f:
    qa_pairs = json.load(f)

rows = []
for qa in qa_pairs[:6]:
    resp = pipe.answer(qa['question'])
    rows.append({
        "question": qa['question'][:60],
        "faithfulness": round(faithfulness(resp.answer, resp.contexts), 2),
        "answer_relevance": round(answer_relevance(resp.answer, qa['question']), 2),
    })

print(pd.DataFrame(rows))

---
## 4. Agentic and Multi-Hop Retrieval

Some questions need information stitched from multiple documents (*"Compare BM25 to dense retrieval and recommend which to use for a code-search system"*). A single retrieval round won't surface everything.

**Multi-hop** retrieval iterates:

1. Retrieve for the user query.
2. Have the LLM produce an *intermediate* sub-query from what's still missing.
3. Retrieve again. Repeat until the model says it's done.

**Agentic RAG** generalizes this: at every step the LLM picks `search`, `tool_call`, or `final_answer`.

In [ ]:
def multi_hop_retrieve(question, pipe, max_hops=3):
    """Toy multi-hop loop: at each hop, treat new keywords as the next sub-query."""
    seen_ids = set()
    all_hits = []
    sub_query = question
    for hop in range(max_hops):
        hits = pipe.retrieve(sub_query, top_k=3)
        new_hits = [h for h in hits if h.chunk_id not in seen_ids]
        if not new_hits:
            break
        all_hits.extend(new_hits)
        seen_ids.update(h.chunk_id for h in new_hits)
        # 'Plan' the next sub-query: use the lowest-ranked retrieved chunk as a seed
        sub_query = new_hits[-1].text.split('.')[0]
        print(f"hop {hop}: {len(new_hits)} new hits, next sub-query: {sub_query[:60]}...")
    return all_hits

hops = multi_hop_retrieve(
    "How can I make a RAG system both fresh and low latency?", pipe
)
print(f"\nTotal unique chunks gathered: {len(hops)}")

---
## 5. Production Considerations

| Concern | Lever |
|---------|-------|
| **Latency** | smaller embedder, fewer candidates, pre-warm caches, stream the answer |
| **Caching** | embeddings, candidate lists, prompt-to-answer pairs (with TTL + version tag) |
| **Freshness** | scheduled re-index; upsert by `doc_id`; time-decayed scoring |
| **Sharding** | partition the vector index across machines; fan-out queries; merge |
| **Cost** | rerank only the top 20–50; cache aggressively; pick a smaller embedder |
| **Security** | per-user filters *before* retrieval; audit log every query |

In [ ]:
# Quick latency profile of each pipeline stage
import time

def time_it(fn, *a, **kw):
    t0 = time.perf_counter()
    out = fn(*a, **kw)
    return out, (time.perf_counter() - t0) * 1000

q = "How does hybrid search combine dense and sparse retrieval?"

_, t_embed = time_it(embedder.encode_query, q)
emb = embedder.encode_query(q)
_, t_dense = time_it(dense.search, emb, 5)
_, t_sparse = time_it(sparse.search, q, 5)
_, t_hybrid = time_it(hybrid.search, q, emb, 5)
_, t_full = time_it(pipe.answer, q)

print(f"embed query : {t_embed:6.2f} ms")
print(f"dense search: {t_dense:6.2f} ms")
print(f"sparse BM25 : {t_sparse:6.2f} ms")
print(f"hybrid+RRF  : {t_hybrid:6.2f} ms")
print(f"full RAG    : {t_full:6.2f} ms (incl. mock-LLM template)")

In [ ]:
# Simple in-memory answer cache
class AnswerCache:
    def __init__(self, ttl_seconds=300):
        self.store = {}
        self.ttl = ttl_seconds

    def get(self, key):
        item = self.store.get(key)
        if not item:
            return None
        ts, val = item
        if time.time() - ts > self.ttl:
            del self.store[key]
            return None
        return val

    def set(self, key, val):
        self.store[key] = (time.time(), val)

cache = AnswerCache()

def cached_answer(query):
    hit = cache.get(query)
    if hit is not None:
        return hit, "cache"
    resp = pipe.answer(query)
    cache.set(query, resp)
    return resp, "fresh"

q = "What is RAG?"
_, src1 = cached_answer(q); _, src2 = cached_answer(q)
print(f"first call: {src1}, second call: {src2}")

---
## 6. Capstone Design

**Build a RAG system for your team's wiki.**

Spec it on paper before you write code. A good design names choices for each of:

1. **Document loaders** — what sources, what metadata, what update cadence?
2. **Chunking** — strategy, target token size, overlap?
3. **Embedder** — open or hosted? Latency budget?
4. **Vector store** — in-process, self-hosted, or managed?
5. **Hybrid?** — BM25 alongside dense?
6. **Reranking** — when does the latency cost pay for itself?
7. **Prompt template** — citation format, refusal behaviour, persona?
8. **Evaluation set** — how many queries, how labeled, who reviews?
9. **Latency / cost budgets** — p50 and p95 targets, $/query target?
10. **Security** — access control, PII redaction, audit logging?

Write a one-page design covering each. Then implement a v0 using the chapter's `RAGPipeline` and the patterns from this notebook.

---
## 7. Key Takeaways

- **Hybrid search** wins on most public benchmarks. Use RRF as the no-tuning fusion baseline.
- **Query rewriting** (HyDE / multi-query / decomposition) recovers recall when the query is short or vague.
- **Faithfulness, answer relevance, context precision/recall** are the four numbers a serious RAG system tracks. Lexical proxies are fine for early dev; LLM-as-judge for prod.
- **Latency, caching, freshness, sharding, cost, security** — each is a discipline. Plan for all six before launch.
- The same `RAGPipeline` you built in Notebook 02 already supports every advanced pattern in this notebook through dependency injection.

Continue to **Chapter 14: Fine-tuning & Adaptation** to learn what to do when retrieval alone is not enough.

---
*Generated by Berta AI*